In [1]:
# Run these small tutorials on CPU; select a GPU before importing JAX for larger runs.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np
import marlax as mx
from marlax.envs import CooperativeGrid, GridState
from marlax.models import QTable
from marlax.methods import QLearning, SARSA, ExpectedSARSA


In [2]:
# 1. Use temporary output for the tutorial; use a fresh runs/... path for real work.
from pathlib import Path
from tempfile import TemporaryDirectory
workspace = TemporaryDirectory(prefix="marlax-tutorial-")
output = Path(workspace.name) / "run"
env = CooperativeGrid(size=3)
params = env.default_params()._replace(time_limit=8)
agent = mx.Agent(QTable())
method = SARSA(epsilon_steps=1024)
config = {"env": {"size": 3, "num_agents": 2}, "params": params._asdict()}
run = mx.train(env, params, agent, method, seed=0, num_envs=16, env_steps=512,
               chunk_steps=32, output=output, config=config)
sorted(str(path.relative_to(output)) for path in output.rglob("*") if path.is_file())


['checkpoint/manifest.json',
 'checkpoint/state.npz',
 'config.json',
 'metrics.jsonl',
 'policy/manifest.json',
 'policy/variables.npz',
 'provenance.json']

In [3]:
# 2. Policy loading needs only its JSON manifest and NumPy table.
policy = mx.load_policy(output / "policy")
_, timestep = env.reset(jax.random.PRNGKey(5), params)
np.testing.assert_array_equal(policy.act(timestep.policy_input, (), None).actions,
                              run.policy.act(timestep.policy_input, (), None).actions)
{"policy_shape": policy.variables.shape, "actions": np.asarray(policy.act(timestep.policy_input, (), None).actions)}


{'policy_shape': (2, 567, 5), 'actions': array([1, 3], dtype=int32)}

In [4]:
# 3. A full checkpoint needs the matching configuration and initialized template.
# It restores random keys, worlds, counters, and SARSA's pending next actions.
template = mx.initialize(env, params, agent, method, seed=0, num_envs=16)
restored = mx.load_checkpoint(output / "checkpoint", template)
continued = mx.train(env, params, agent, method, state=restored, env_steps=512, chunk_steps=32)
uninterrupted = mx.train(env, params, agent, method, seed=0, num_envs=16, env_steps=1024, chunk_steps=32)
for actual, expected in zip(jax.tree.leaves(continued.state), jax.tree.leaves(uninterrupted.state)):
    np.testing.assert_array_equal(actual, expected)
{"restored_env_steps": int(restored.method_state.env_steps),
 "continued_env_steps": int(continued.state.method_state.env_steps), "same_state": True}


{'restored_env_steps': 512, 'continued_env_steps': 1024, 'same_state': True}

In [5]:
# 4. Save a standalone policy or a low-level checkpoint without creating a run.
mx.save_policy(Path(workspace.name) / "policy", continued.policy, env.spec)
mx.save_checkpoint(Path(workspace.name) / "checkpoint", continued.state)
# The tutorial's temporary files are no longer needed.
workspace.cleanup()
